# "But wait... there's more"

## A More Visible Agent Loop

The Digital Twin contained an Agent Loop. But it was behind-the-scenes, running every time the user asked a message. Using its tools and then replying. It didn't feel very... loopy.

### Adding 2 more ingredients to make it more real

Let's make an Agent Loop with some familiar features borrowed from Claude Code:

1. A Terminal UI (TUI)
2. A Checklist tool to cause and track multiple tool calls


In [ ]:
from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json
import os

load_dotenv(override= True)

gemini_api_key = os.getenv("GOOGLE_API_KEY")
gemini_base_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
gemini_model = "gemini-2.5-flash-lite"
gemini = OpenAI(api_key=gemini_api_key, base_url=gemini_base_url)

openrouter_api_key = os.getenv("OPENROUTER_API_KRY")
openrouter_base_url = "https://openrouter.ai/api/v1"
openrouter = OpenAI(api_key=openrouter_api_key, base_url= openrouter_base_url)
openrouter_model = "cohere/north-mini-code:free"

checklist = []
completed = []

In [3]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [4]:
def get_checklist_report():
    result = ""
    for index, item in enumerate(checklist):
        if completed[index]:
            result += f"[green]✓ [{index + 1}]  {item} [/green]\n"
        else:
            result += f"{index + 1} : {item}  \n"
    show(result)
    return result

In [ ]:
def create_checklist(descriptions: list[str]) -> str:
    checklist.extend(descriptions)
    completed.extend([False] * len(checklist))
    result = get_checklist_report()

def mark_completions(index, completion_note):
    if  1 <= index <= len(checklist):
        completed[index-1] = True
    else:
        print(f"No checklist at this index {index}")
    Console().print(completion_note)
    result = get_checklist_report()

In [ ]:
create_checklist_json = {
    "name": "create_checklist",
    "description": "get a list of descriptions and create a new checklist from them and return the created checklist",
    "parameters": {
        "type": "object",
        "properties": {
            "description": {
                "type": "array",
                "items": {"type": "string"},
                "description": "Descriptions of checklist items"
            }
        }
    },
    "required": ["descriptions"],
    "additionalProperties": False
}


mark_completions_json = {
    "name": "mark_completions",
    "description": "Mark complete the checklist item at the given position (starting from 1) and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "index": {
                "type": "integer",
                "description": "checklist item that is completed"
            },
            "completion_note": {
                "type": "string",
                "description": "completion note of the given item of checklist"
            }
        }
    },
    "required": ["index", "completion_note"],
    "additionalProperties": False
}

In [10]:
tools = [{"type": "function", "function": create_checklist_json},
         {"type": "function", "function": mark_completions_json}]

tools_map = {
    "create_checklist": create_checklist,
    "mark_completions": mark_completions
}

In [ ]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        args = json.loads(tool_call.function.arguments)
        tool = tools_map.get(tool_call)
        result = tool(**args) if tool else {} 
        results.append([{"role": "tool", "content": json.dumps(result), "tool_call_id": tool_call.id}])
    return results

def loop(messages):    
    response = openrouter.chat.completions.create(model=openrouter_model, messages=messages)
    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        tool_calls = message.tool_calls
        results = handle_tool_calls(tool_calls)
        messages.append(message)
        messages.extend(results)
        response = openrouter.chat.completions.create(model=openrouter_model, messages=messages)
    return response.choices[0].message.content

In [12]:
system_message = """
You are given a problem to solve, by using your checklist tools to plan a list of steps, then carrying out each step in turn.
Now create a plan, set the checklist, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """"
A train leaves Boston at 2:00 pm traveling 60 mph.
Another train leaves New York at 3:00 pm traveling 80 mph toward Boston.
When do they meet?
"""
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

In [ ]:
result = loop(messages)
show(result)